# Kandinsky V2A — Full Lite 15-Second Window Consistency

## Experiment question

Does the full Lite checkpoint keep sound quality, event relevance, and noise level consistent across a 15-second clip when generation is split into three consecutive five-second windows?

- One input video, split into **0–5, 5–10, and 10–15 seconds**.
- Full Lite checkpoint for all windows.
- Same short event prompt and seed for all three.
- No training, fine-tuning, or new dataset.
- The three WAVs are also concatenated without fades so boundary changes are easy to hear.

This evaluates window-to-window consistency. It does not test direct game-token conditioning or produce separated event stems.

## 1 — Check Colab and select storage

Use the same A100 80 GB runtime as the recent Lite comparison. The notebook prefers the Colab local-scratch disk when it has more free space.

In [ ]:
import os, sys, shutil, subprocess, json, time, hashlib, zipfile
from pathlib import Path
from datetime import datetime, timezone

print("Python:", sys.version)
if sys.version_info < (3, 13):
    raise RuntimeError("This notebook expects the Python 3.13+ Colab runtime used for Kandinsky.")

if not shutil.which("nvidia-smi"):
    raise RuntimeError("GPU runtime is missing. Select an A100 GPU and reconnect.")

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True, check=True
)
print("GPU:", gpu.stdout.strip())

roots = [Path("/content/local-scratch"), Path("/content")]
CACHE_ROOT = max(
    (p for p in roots if p.exists()),
    key=lambda p: shutil.disk_usage(p).free
)
free_gib = shutil.disk_usage(CACHE_ROOT).free / (1024 ** 3)
print(f"Model storage: {CACHE_ROOT} | free: {free_gib:.1f} GiB")
if free_gib < 35:
    raise RuntimeError("At least 35 GiB free space is recommended for the full Lite checkpoint.")

os.environ["HF_HOME"] = str(CACHE_ROOT / "huggingface")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE_ROOT / "huggingface" / "hub")
os.environ["KANDINSKY_HOME"] = str(CACHE_ROOT / "kandinsky_home")

WORK = Path("/content/kandinsky_full_lite_15s")
INPUT_DIR = WORK / "input"
WINDOW_DIR = WORK / "windows"
OUTPUT_DIR = WORK / "outputs"
for directory in (INPUT_DIR, WINDOW_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

## 2 — Install the pinned Kandinsky environment and download full Lite

The earlier quality-comparison report used Kandinsky source commit `7ceeefad7fb625011f7db602c43893a989f19c15`. This notebook pins the same commit. It installs the official consumer dependencies directly and skips optional SageAttention compilation because generation uses PyTorch SDPA.

In [ ]:
REPO = Path("/content/kandinsky-6")
PINNED_COMMIT = "7ceeefad7fb625011f7db602c43893a989f19c15"

if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "https://github.com/kandinskylab/kandinsky-6.git", str(REPO)],
        check=True
    )
subprocess.run(["git", "-C", str(REPO), "checkout", PINNED_COMMIT], check=True)

subprocess.run([sys.executable, "-m", "pip", "-q", "install", "uv"], check=True)
if not Path("/usr/local/bin/just").exists():
    subprocess.run(
        ["bash", "-lc", "curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin"],
        check=True
    )

setup = subprocess.run(
    ["uv", "sync", "--extra", "consumer", "--inexact", "--no-dev"],
    cwd=REPO, capture_output=True, text=True
)
print(setup.stdout[-5000:])
if setup.returncode:
    print(setup.stderr[-7000:])
    raise RuntimeError("Dependency setup failed. The uv output above contains the underlying error.")

SOURCE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
print("Pinned source commit:", SOURCE_COMMIT)

subprocess.run(["just", "download", "lite"], cwd=REPO, check=True)
print("Full Lite checkpoint is ready.")

## 3 — Provide the same 15-second source video and prompt

Use the 15-second video from the previous window experiment. If it is already available at `/content/kandinsky_v2a_input.mp4`, the notebook reuses it; otherwise Colab asks you to upload it once.

Keep the prompt short and unchanged across all three windows. The default below is `footsteps`, matching the earlier test; change it only if your source video has a different main sound event.

In [ ]:
from IPython.display import Audio, Video, display, Markdown

def require_existing_file(value, label):
    """Reject an unset or missing path with a direct, actionable message."""
    if value is None:
        raise ValueError(f"{label} path is None. Run this notebook's setup/input cells in order.")
    if not isinstance(value, (str, os.PathLike)):
        raise TypeError(f"{label} must be a path, got {type(value).__name__}.")
    path = Path(value).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path

DEFAULT_SOURCE_VIDEO = Path("/content/kandinsky_v2a_input.mp4")
SOURCE_VIDEO = DEFAULT_SOURCE_VIDEO
if SOURCE_VIDEO is None:
    SOURCE_VIDEO = DEFAULT_SOURCE_VIDEO
elif not isinstance(SOURCE_VIDEO, (str, os.PathLike)):
    raise TypeError(f"Source video must be a path, got {type(SOURCE_VIDEO).__name__}.")
else:
    SOURCE_VIDEO = Path(SOURCE_VIDEO).expanduser()

if not SOURCE_VIDEO.is_file():
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No source video was uploaded. Run this input cell again.")
    uploaded_name = next(iter(uploaded))
    SOURCE_VIDEO = Path("/content") / Path(uploaded_name).name
    SOURCE_VIDEO.write_bytes(uploaded[uploaded_name])

SOURCE_VIDEO = require_existing_file(SOURCE_VIDEO, "Source video")

PROMPT = "footsteps"
SEED = 42
WINDOW_SECONDS = 5
WINDOW_STARTS = [0, 5, 10]

def run_command(command, cwd=None, tail=2200):
    result = subprocess.run(command, cwd=cwd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout[-tail:])
    if result.returncode:
        if result.stderr:
            print(result.stderr[-5000:])
        raise RuntimeError("Command failed; inspect the output above.")
    return result

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

probe = run_command([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(SOURCE_VIDEO)
])
SOURCE_DURATION = float(probe.stdout.strip())
if SOURCE_DURATION < 14.95:
    raise ValueError(f"Expected a 15-second video; found {SOURCE_DURATION:.2f} seconds.")

print(f"Source: {SOURCE_VIDEO.name} | {SOURCE_DURATION:.2f}s")
print("Prompt:", PROMPT, "| Seed:", SEED)
display(Video(filename=str(require_existing_file(SOURCE_VIDEO, "Source video")), embed=True))

## 4 — Prepare three muted five-second windows

Each window keeps its original position in the 15-second source. Original video audio is discarded so the model generates a fresh track from the video and prompt.

In [ ]:
WINDOWS = []
for start in WINDOW_STARTS:
    window_path = WINDOW_DIR / f"window_{start:02d}_{start+WINDOW_SECONDS:02d}.mp4"
    run_command([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-ss", str(start), "-i", str(SOURCE_VIDEO), "-t", str(WINDOW_SECONDS),
        "-map", "0:v:0", "-an",
        "-vf", "fps=24,scale='min(960,iw)':-2",
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "25",
        "-pix_fmt", "yuv420p", str(window_path)
    ])
    WINDOWS.append({
        "start_seconds": start,
        "end_seconds": start + WINDOW_SECONDS,
        "path": window_path,
        "sha256": sha256_file(window_path),
    })
    print(f"Prepared {start:02d}–{start+WINDOW_SECONDS:02d}s:", window_path.name)


## 5 — Generate each window sequentially with full Lite

The checkpoint and all inference settings remain fixed. Only the video window changes. The A100 full Lite run took about ten minutes per five-second output in the previous comparison, so allow roughly half an hour for these three generations, plus setup and downloads.

In [ ]:
DEVICE_CONFIG = REPO / "full-lite-window.yaml"
DEVICE_CONFIG.write_text(
    """checkpoint: lite
cache:
  mode: none
offload:
  strategy: none
compile:
  strategy: none
attention:
  engine: sdpa
text_embedder:
  quantized_qwen: false
beautifier:
  name: none
""",
    encoding="utf-8"
)

CANDIDATES = []
for window in WINDOWS:
    start = window["start_seconds"]
    stem = f"full_lite_{start:02d}_{start+WINDOW_SECONDS:02d}"
    output_mp4 = OUTPUT_DIR / f"{stem}.mp4"
    output_wav = OUTPUT_DIR / f"{stem}.wav"

    command = [
        "just", "generate", PROMPT,
        "--config", str(DEVICE_CONFIG),
        "--attention-engine", "sdpa",
        "--video-file", str(window["path"]),
        "--time-length", str(WINDOW_SECONDS),
        "--out", str(output_mp4),
        "--seed", str(SEED),
    ]

    print(f"Generating source window {start}–{start+WINDOW_SECONDS}s with full Lite...")
    started = time.time()
    run_command(command, cwd=REPO, tail=3000)
    output_mp4 = require_existing_file(output_mp4, f"Generated MP4 for {start}–{start+WINDOW_SECONDS}s")
    elapsed = round(time.time() - started, 2)

    run_command([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
        "-i", str(output_mp4), "-vn", "-ac", "2", "-ar", "44100",
        "-c:a", "pcm_s16le", str(output_wav)
    ])

    CANDIDATES.append({
        "source_window_start_seconds": start,
        "source_window_end_seconds": start + WINDOW_SECONDS,
        "prompt": PROMPT,
        "seed": SEED,
        "checkpoint": "lite",
        "generation_seconds": elapsed,
        "input_video_sha256": window["sha256"],
        "output_mp4": output_mp4.name,
        "output_wav": output_wav.name,
        "output_mp4_sha256": sha256_file(output_mp4),
        "output_wav_sha256": sha256_file(output_wav),
    })
    print(f"Finished in {elapsed:.1f}s")
print("Completed windows:", len(CANDIDATES), "/", len(WINDOWS))

## 6 — Listen to the windows individually

Listen for event relevance, noise, and sync in each segment. In particular, compare the final window with the first two: the earlier experiment suggested that the last segment can drift from the scene.

In [ ]:
for item in CANDIDATES:
    start = item["source_window_start_seconds"]
    end = item["source_window_end_seconds"]
    display(Markdown(f"### Source window {start}–{end}s | generated in {item['generation_seconds']:.0f}s"))
    output_mp4 = require_existing_file(OUTPUT_DIR / item["output_mp4"], "Generated window video")
    output_wav = require_existing_file(OUTPUT_DIR / item["output_wav"], "Generated window audio")
    display(Video(filename=str(output_mp4), embed=True))
    display(Audio(filename=str(output_wav)))

## 7 — Concatenate the three WAVs without crossfades

This creates a simple 15-second listening preview. There is no overlap, fade, loudness normalization, or other post-processing, so any boundary jump remains audible and traceable to the generated windows.

In [ ]:
CONCAT_LIST = WORK / "concat_list.txt"
CONCAT_LIST.write_text(
    "\n".join(f"file '{(OUTPUT_DIR / item['output_wav']).as_posix()}'" for item in CANDIDATES) + "\n",
    encoding="utf-8"
)
FULL_WAV = WORK / "full_lite_15s_concatenated.wav"
run_command([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
    "-f", "concat", "-safe", "0", "-i", str(CONCAT_LIST),
    "-c:a", "pcm_s16le", str(FULL_WAV)
])
FULL_WAV = require_existing_file(FULL_WAV, "Concatenated audio")
print("Concatenated length: 15 seconds")
display(Audio(filename=str(FULL_WAV)))

## 8 — Record listening notes and download the report

The ratings are your listening observations, not automatic quality metrics. The source video is intentionally excluded from the ZIP; the ZIP contains only generated outputs and the report.

In [ ]:
SCORES = {
    f"{item['source_window_start_seconds']:02d}_{item['source_window_end_seconds']:02d}": {
        "audio_quality_1_to_5": None,
        "naturalness_1_to_5": None,
        "artifacts_or_noise_1_to_5": None,  # 1 = very little; 5 = very prominent
        "event_match_1_to_5": None,
        "sync_1_to_5": None,
        "continuity_with_previous_window_1_to_5": None,
        "notes": "",
    }
    for item in CANDIDATES
}

REPORT = {
    "experiment": "kandinsky_v2a_full_lite_15s_window_consistency_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_video_name": SOURCE_VIDEO.name,
    "source_video_sha256": sha256_file(SOURCE_VIDEO),
    "source_video_duration_seconds": SOURCE_DURATION,
    "source_commit": SOURCE_COMMIT,
    "gpu": gpu.stdout.strip(),
    "checkpoint": "kandinskylab/Kandinsky-6.0-Lite-5s-Diffusers",
    "prompt": PROMPT,
    "seed": SEED,
    "window_seconds": WINDOW_SECONDS,
    "candidates": CANDIDATES,
    "listening_scores": SCORES,
    "concatenated_audio": FULL_WAV.name,
    "notes": "Audio concatenated in timeline order with no crossfades or loudness processing."
}
REPORT_PATH = WORK / "full_lite_15s_report.json"
REPORT_PATH.write_text(json.dumps(REPORT, ensure_ascii=False, indent=2), encoding="utf-8")

ZIP_PATH = Path("/content/kandinsky_v2a_full_lite_15s_report.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(REPORT_PATH, "full_lite_15s_report.json")
    archive.write(FULL_WAV, "full_lite_15s_concatenated.wav")
    for item in CANDIDATES:
        archive.write(OUTPUT_DIR / item["output_wav"], "windows/" + item["output_wav"])
        archive.write(OUTPUT_DIR / item["output_mp4"], "windows/" + item["output_mp4"])

ZIP_PATH = require_existing_file(ZIP_PATH, "Report ZIP")
print("Report ZIP:", ZIP_PATH, "|", round(ZIP_PATH.stat().st_size / 1e6, 1), "MB")
from google.colab import files
files.download(str(ZIP_PATH))